In [0]:
# === Init cell with config (RUN FIRST) ===
from pyspark.sql.functions import col, count, avg, max, min, sum, round, concat_ws, lit, when

storage_account_hierarchical = "foremstorageaccount"
silver_adsl_path = f"abfss://adf-silver@{storage_account_hierarchical}.dfs.core.windows.net/"
gold_adsl_path = f"abfss://adf-gold@{storage_account_hierarchical}.dfs.core.windows.net/"

df_silver = spark.read.format("delta").load(silver_adsl_path)
# filter articles published in 11.2022
df_silver = df_silver.filter((col("published_at").cast("date") >= "2022-12-01"))

print(f"Number of entries: {df_silver.count()}")


In [0]:
from pyspark.sql.functions import col, when

conditionComments = ((col("comments_count") > 0) & (col("comments_count").isNotNull()))
conditonPublicReactions = ((col("public_reactions_count") > 0) & (col("public_reactions_count").isNotNull()))

# Articles with comments OR reactions (either one or both)
df_gold_articles_fact_table = (
    df_silver
    .withColumn("has_comments", when(conditionComments, 1).otherwise(0))
    .withColumn("has_reactions", when(conditonPublicReactions, 1).otherwise(0))
    .select(
        "id",
        "published_at",
        "comments_count",
        "public_reactions_count",
        "has_comments",
        "has_reactions",
        "language",
        "user_id",
        "reading_time_minutes",
        "year",
        "month"
    )
)
# df_gold_articles_fact_table.limit(10).display()

df_gold_articles_fact_table.write.format("delta") \
    .partitionBy("year", "month") \
    .mode("overwrite") \
    .save(f"{gold_adsl_path}articles_fact")

In [0]:
from pyspark.sql.functions import explode, col, count, row_number, desc

df_tags_fact_table = df_silver.select(
    "id",
    "reading_time_minutes",
    "comments_count",
    "public_reactions_count",
    "user_id",
    "date",
    "year",
    "month",
    explode(col("tag_list")).alias("tag"),
).filter(col("tag").isNotNull() & (col("tag") != "")) # filter articles with no tags

df_tags_fact_table.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_adsl_path}tags_fact")


In [0]:
from pyspark.sql.functions import col, explode, count, desc
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# 1️⃣ Explode tags once
df_tags = (
    df_silver
    .select("year", "month", explode(col("tag_list")).alias("tag"))
    .filter(col("tag").isNotNull() & (col("tag") != ""))
)

# 2️⃣ Articles per tag per month
df_tag_month_2 = (
    df_tags
    .groupBy("year", "month", "tag")
    .agg(count("*").alias("articles_count"))
)

# 3️⃣ Total articles per month (from non-exploded table)
df_total_month_2 = (
    df_silver
    .groupBy("year", "month")
    .agg(count("*").alias("total_articles_month"))
)

# 4️⃣ Find Top 10 tags PER MONTH
window_month = Window.partitionBy("year", "month").orderBy(desc("articles_count"))

df_top10_each_month = (
    df_tag_month_2
    .withColumn("rank", row_number().over(window_month))
    .filter(col("rank") <= 100)
    .drop("rank")
)

# 5️⃣ Create UNION list of tags that appeared in any monthly Top 10
top_tags_union = (
    df_top10_each_month
    .select("tag")
    .distinct()
)

# 6️⃣ Keep only those tags across ALL months
df_tag_filtered = (
    df_tag_month_2
    .join(top_tags_union, on="tag", how="inner")
)

# 7️⃣ Join totals and compute percentage
df_final = (
    df_tag_filtered
    .join(df_total_month_2, ["year", "month"], "inner")
    .withColumn(
        "percentage",
        (col("articles_count") / col("total_articles_month")) * 100
    )
    .orderBy("year", "month", desc("articles_count"))
)

df_final.display()

df_final.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_adsl_path}tags_per_month_fact")

In [0]:
from pyspark.sql.functions import col, when

conditionComments = ((col("comments_count") > 0) & (col("comments_count").isNotNull()))
conditonPublicReactions = ((col("public_reactions_count") > 0) & (col("public_reactions_count").isNotNull()))

# Articles with comments OR reactions (either one or both)
df_gold_full_articles_fact_table = (
    df_silver
    .withColumn("has_comments", when(conditionComments, 1).otherwise(0))
    .withColumn("has_reactions", when(conditonPublicReactions, 1).otherwise(0))
    .select(
        "id",
        "title",
        "created_at",
        "published_at",
        "user",
        "tag_list",
        "positive_reactions_count",
        "public_reactions_count",
        "comments_count",
        "reading_time_minutes",
        "language",
        "collection_id",
        "batch_id",
        "batch_ingestion_date",
        "has_comments",
        "has_reactions",
        "user_username",
        "user_name",
        "user_id",
        "date",
        "year",
        "month",
        "day"
    )
)
# df_gold_articles_fact_table.limit(10).display()

df_gold_full_articles_fact_table.write.format("delta") \
    .partitionBy("year", "month") \
    .mode("overwrite") \
    .save(f"{gold_adsl_path}full_articles_fact")
